---
**`NOTEBOOK 01 / 03`** · *Causal Inference Series*

# A/B Testing Playbook — Cookie Cats

*Tactile Entertainment · 90,189 players · Completed randomized experiment dataset*

| | |
|---|---|
| Method | A/B Test / Randomized Controlled Trial |
| Control | `gate_30` |
| Treatment | `gate_40` |
| Primary metric | `retention_7` — binary / proportion |
| Randomization unit | Player |
| Primary test | Two-proportion Z-test |
| Observed effect | `gate_40 - gate_30 ≈ -0.82pp` |
| 95% CI | approximately `[-1.33pp, -0.31pp]` |
| p-value | approximately `0.0016` |
| SRM | Target allocation is not stored in the public file; assuming 50/50 triggers an alert |

---

Cookie Cats is a mobile puzzle game in which players encounter progression gates.

The experiment compares the existing gate at level 30 with a version that moves it to level 40.

**Causal question:** does moving the gate from level 30 to level 40 change Day-7 retention?

Randomization makes treatment and control comparable on average, so a valid randomized experiment can identify the causal effect of **assignment** to the new experience.

---

## How to use this notebook

This notebook is both:

1. a **Cookie Cats case study**, and
2. a **general A/B testing guide** for product, fintech, gaming, e-commerce, subscription and marketplace experiments.

### A/B testing lifecycle

| Step | What happens | Timing |
|---|---|---|
| `0` Setup | Load data and understand schema | Before analysis |
| `1` Hypotheses & metrics | Define H0/H1, primary metric, guardrails and success criteria | Before launch |
| `2` Randomization unit | Decide who/what is randomized | Before launch |
| `3` Sample size & duration | Baseline, MDE, alpha, power, traffic, seasonality, maturation | Before launch |
| `4` Pre-analysis plan | Test, CLT checks, CUPED, sequential rules, multiple testing | Before launch |
| `5` Launch & collect data | Create control/treatment assignment operationally and collect outcomes | Experiment running |
| `6` Sanity checks | SRM, exposure, tracking, missingness, contamination | From first assignments onward |
| `7` Statistical analysis | Estimate effect, CI and p-value using the pre-specified test | After planned horizon / maturation |
| `8` Business decision | Combine effect size, uncertainty, guardrails and validity | After analysis |
| `9` Advanced extensions | CUPED, HTE, multiple testing details | When justified |
| `10` Common mistakes | Production + interview checklist | Always |

> **Step 5 = run the product experiment.**  
> **Step 7 = run the statistical test.**

---


## 0. Setup

Load the dataset and understand its structure.

At this stage:

- inspect columns
- inspect missing values and duplicates
- verify the analysis unit
- identify the treatment-label column
- **do not yet split into control/treatment dataframes**
- **do not inspect treatment-effect results**

The control/treatment split belongs in Step 5 because it represents the realized assignment produced by the experiment.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import chisquare, norm, ttest_ind
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest

import warnings
warnings.filterwarnings("ignore")

candidates = [
    Path("../data/cookie_cats.csv"),
    Path("data/cookie_cats.csv"),
    Path("cookie_cats.csv"),
]

data_path = next((p for p in candidates if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "cookie_cats.csv not found. Put it in ../data/, data/, or the notebook folder."
    )

df = pd.read_csv(data_path)

print(f"Shape           : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Missing values  : {df.isnull().sum().sum():,}")
print(f"Duplicate users : {df['userid'].duplicated().sum():,}")
print(f"Columns         : {list(df.columns)}")
print(f"Version labels  : {sorted(df['version'].dropna().unique())}")

display(df.head())

Shape           : 90,189 rows x 5 columns
Missing values  : 0
Duplicate users : 0
Columns         : ['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']
Version labels  : ['gate_30', 'gate_40']


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


### Setup interpretation

For Cookie Cats:

- one row should represent one player
- `userid` is the analysis identifier
- `version` stores the realized experiment assignment
- `retention_1` and `retention_7` are binary outcomes
- `sum_gamerounds` is measured after assignment

If one randomized user contributed many dependent rows, a simple independent-observation test would not be enough; standard errors would need to account for clustering / repeated measures.


## 1. Hypotheses, Primary Metric & Guardrails

Define success **before seeing the treatment result**.

### Metric roles

| Role | Purpose | Examples |
|---|---|---|
| **Primary metric** | Main decision metric | retention, conversion, revenue/user |
| **Guardrail** | Metric that must not deteriorate beyond an acceptable limit | fraud, crashes, complaints, cancellations |
| **Secondary metric** | Helps explain mechanism / side effects | D1 retention, engagement, funnel steps |

### Cookie Cats

**Primary metric:** `retention_7`

- `1` = player retained on Day 7
- `0` = player not retained on Day 7

**Secondary metric:** `retention_1`

The public file does **not document an official guardrail metric**.  
`sum_gamerounds` can be explored as a **secondary engagement outcome**, but we should not retroactively call it an official guardrail.

### Hypotheses

For a two-sided test:

- **H0:** D7 retention is equal under `gate_30` and `gate_40`
- **H1:** D7 retention is different under `gate_30` and `gate_40`

Use a two-sided test when both improvement and deterioration matter.

Use a one-sided test only when the direction is justified **before** seeing the data.

> Do not change the primary metric, hypothesis direction or success threshold after looking at results.


## 2. Randomization Unit

Choose the unit at which treatment is assigned.

| Situation | Typical unit | Why |
|---|---|---|
| Persistent user experience | User / account | Avoid showing both variants to one user |
| Session-only change with no carryover | Session | Treatment is temporary |
| Order-level intervention | Order | Only if repeated exposure is acceptable |
| Store / city / marketplace change | Store / region / market | Reduce spillovers |
| Social / network effects | Cluster / network group | Individual assignment may create interference |

### Cookie Cats

Randomization unit = **player (`userid`)**

Why:

- the gate persists across sessions
- session-level assignment could contaminate the experience
- D7 retention is measured at player level

### Senior-level checks

Also ask:

- Can treatment spill over from treated to control units?
- Can the same person appear under multiple IDs/devices?
- If randomizing clusters, do we need a larger sample because of intra-cluster correlation?

**Rule:** randomize at a level that prevents contamination and matches the dependence structure of the outcome.


## 3. Sample Size & Experiment Duration

Plan sample size **before launch**.

### Inputs

| Input | Meaning |
|---|---|
| **Baseline** | Historical value of the primary metric |
| **MDE** | Smallest effect worth reliably detecting |
| **Alpha** | Type I error / false-positive rate |
| **Power** | Probability of detecting the planned effect if it is real |
| **Beta** | `1 - power` |

Illustrative planning assumptions:

- Historical D7 retention: **19%**
- MDE: **1 percentage point**
- Alpha: **5%**
- Power: **80%** → Beta = **20%**

### Power

80% power means:

> If the true effect is the effect size used in the power calculation, the planned experiment would detect it about 80% of the time across repeated experiments.

It does **not** mean 20% of business decisions will be wrong.

### Duration is not just sample size

Also consider:

- traffic / recruitment speed
- weekday / weekend or seasonal cycles
- outcome maturation
- novelty effects
- delayed outcomes
- fixed-horizon vs sequential stopping

For D7 retention, the final cohort needs enough time to reach Day 7 before the metric is mature.


In [ ]:
# Sample-size planning for a binary / proportion primary metric

baseline = 0.19
mde = 0.01
alpha = 0.05
power = 0.80

effect_size = abs(proportion_effectsize(baseline, baseline + mde))

n_required = NormalIndPower().solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1.0,
    alternative="two-sided",
)

n_required = int(np.ceil(n_required))

print(f"Historical baseline : {baseline:.1%}")
print(f"MDE                 : {mde*100:.1f} percentage point")
print(f"Alpha               : {alpha:.2f}")
print(f"Power               : {power:.0%}")
print(f"Required per group  : {n_required:,}")

Historical baseline : 19.0%
MDE                 : 1.0 percentage point
Alpha               : 0.05
Power               : 80%
Required per group  : 24,638


### Sample-size interpretation

With these illustrative assumptions, the required sample is roughly **24.6k players per arm**.

Important:

- use a historical / pre-experiment baseline
- do not choose the MDE from the observed experiment result
- if cluster randomization is used, a simple individual-level sample-size formula is not sufficient
- actual experiment duration cannot be recovered from this public file because it does not include assignment dates


## 4. Pre-analysis Plan

Choose the analysis **before reading the treatment effect**.

### 4.1 Test-selection guide

| Metric / estimand | Example | Typical approach | Key caution |
|---|---|---|---|
| **Binary / proportion** | retention, conversion | Two-proportion Z-test / logistic regression | Repeated attempts may require clustered SEs |
| **Rare binary** | rare fraud event | Exact methods / Fisher in suitable small tables | Normal approximation may fail |
| **Continuous mean** | revenue/user, time/user | Welch t-test | Heavy tails / outliers can reduce precision |
| **Highly skewed continuous mean** | spend/user | Welch + bootstrap sensitivity | Log transform changes what is estimated |
| **Count per unit** | orders/user | Mean comparison / bootstrap; Poisson or NegBin model if appropriate | Count does not automatically imply Poisson |
| **Rate with exposure** | crashes/user-hour | Poisson / NegBin with exposure offset | Exposure must be modeled |
| **Ratio metric** | revenue/order | Delta method / bootstrap | Numerator and denominator are dependent |
| **Time-to-event** | time to churn | Log-rank / Cox | Censoring matters |
| **Repeated observations** | many sessions/user | Cluster-robust SE / GEE / mixed model | Rows are not independent |

### Key principle

Do not choose the test only from the shape of the raw column.

Ask:

1. What is the **estimand**?
2. What is the **randomization unit**?
3. Are observations **independent**?
4. What uncertainty estimator matches that structure?


### 4.2 CLT — when is a normal approximation reasonable?

The **Central Limit Theorem (CLT)** is about the sampling distribution of an estimator. It does not mean the raw data become normal.

For a binary / proportion metric, check that each arm has enough expected successes and failures.

Useful rule of thumb:

- `n × p >= 10`
- `n × (1 - p) >= 10`

Example:

- `n = 1,000`, `p = 0.20`
- successes ≈ 200
- failures ≈ 800
- normal approximation is reasonable

But:

- `n = 100`, `p = 0.01`
- successes ≈ 1
- normal approximation is poor even though `n > 30`

For continuous means there is **no universal `n > 30` rule**. More skewness, heavy tails and extreme outliers generally require more care.

### CLT does NOT fix

- dependent observations
- repeated users treated as independent
- bad randomization
- SRM
- wrong exposure definitions
- an inappropriate Poisson assumption


### 4.3 Cookie Cats analysis plan

Primary outcome:

- `retention_7`
- one binary outcome per randomized player
- large sample

Primary test:

**Two-proportion Z-test**

Other decisions to pre-specify:

| Topic | Plan |
|---|---|
| **Two-sided / one-sided** | Two-sided here |
| **Peeking** | Do not repeatedly stop a fixed-horizon test when p first falls below 0.05 |
| **Sequential testing** | If interim decisions are required, use a formal group-sequential / alpha-spending method |
| **CUPED** | Only with a true pre-treatment covariate |
| **Multiple testing** | Predefine confirmatory metrics; use Holm/Bonferroni/FDR when appropriate |
| **HTE / subgroups** | Pre-specify key segments or label broad subgroup mining exploratory |
| **Analysis population** | Primary randomized analysis should follow assignment (ITT principle) |

> Do not invent sequential alpha thresholds manually.


## 5. Launch the Experiment & Create the Realized Arms

This is the **operational experiment step**, not the statistical test.

In production:

1. deploy control and treatment
2. randomly assign eligible units
3. store the assignment
4. log whether treatment was actually delivered
5. collect outcomes
6. keep definitions stable during the planned experiment
7. allow outcomes to mature

### Important: assignment vs exposure

The primary randomized analysis normally follows **assignment** (intention-to-treat logic).

If some assigned users fail to receive the experience, do not simply drop them; doing so can destroy randomization. Investigate delivery and pre-specify any non-compliance analysis separately.

### Cookie Cats

The experiment has already happened. The `version` column contains the realized assignment.

So **this** is the correct place to create the control and treatment dataframes.


In [ ]:
# Realized randomized arms from the completed experiment

control = df[df["version"] == "gate_30"].copy()
treatment = df[df["version"] == "gate_40"].copy()

print("--- Realized experiment arms ---")
print(f"Control   gate_30 : {len(control):,} players")
print(f"Treatment gate_40 : {len(treatment):,} players")
print(f"Total             : {len(control) + len(treatment):,} players")

# Important:
# This code does NOT randomize users now.
# It separates the observations according to the assignment produced by the real experiment.

--- Realized experiment arms ---
Control   gate_30 : 44,700 players
Treatment gate_40 : 45,489 players
Total             : 90,189 players


### Step 5 interpretation

Different realized sample sizes are **not automatically an error**.

A random 50/50 process does not guarantee exactly equal counts.

The next step is to test whether the observed difference is plausible under the experiment's **configured target allocation**.


## 6. Sanity Checks — SRM, Tracking & Data Quality

These checks begin **after assignments start appearing**, often long before the experiment finishes.

### Before launch vs after launch

| Before launch | Once experiment starts |
|---|---|
| Validate randomization code | Check observed allocation |
| Validate event logging | SRM |
| Confirm eligibility rules | Missingness / logging failures |
| Confirm target allocation | Exposure failures |
| Optional A/A test | Cross-over / contamination |

### SRM — Sample Ratio Mismatch

SRM asks:

> Is the observed treatment/control allocation compatible with the configured allocation ratio?

If target allocation is 50/50:

- H0: observed counts are compatible with 50/50
- a very small p-value raises an SRM alert

If SRM appears, investigate:

- assignment service
- eligibility filters
- exposure failures
- logging loss
- platform/version-specific bugs
- post-randomization filtering

**Never delete observations or downsample an arm just to force a 50/50 split.**


In [ ]:
print("--- Basic integrity checks ---")
print(f"Missing values     : {df.isnull().sum().sum():,}")
print(f"Duplicate user IDs : {df['userid'].duplicated().sum():,}")

# The public file does not contain the experiment configuration.
# 50/50 is used ONLY as an illustrative assumption.
expected_control_share = 0.50

observed = np.array([len(control), len(treatment)])
expected = np.array([
    len(df) * expected_control_share,
    len(df) * (1 - expected_control_share),
])

chi2_srm, p_srm = chisquare(f_obs=observed, f_exp=expected)

print("\n--- Illustrative SRM check: assuming target = 50/50 ---")
print(f"Observed gate_30 : {observed[0]:,}")
print(f"Observed gate_40 : {observed[1]:,}")
print(f"Chi-squared      : {chi2_srm:.4f}")
print(f"p-value          : {p_srm:.4f}")

if p_srm < 0.05:
    print("SRM ALERT under the 50/50 assumption.")
    print("Verify the true configured allocation before trusting this diagnosis.")
else:
    print("No SRM alert under the specified allocation.")

--- Basic integrity checks ---
Missing values     : 0
Duplicate user IDs : 0

--- Illustrative SRM check: assuming target = 50/50 ---
Observed gate_30 : 44,700
Observed gate_40 : 45,489
Chi-squared      : 6.9024
p-value          : 0.0086
SRM ALERT under the 50/50 assumption.
Verify the true configured allocation before trusting this diagnosis.


### SRM interpretation for this public dataset

Observed group sizes are approximately:

- `gate_30`: **44,700**
- `gate_40`: **45,489**

Assuming a true 50/50 target gives an SRM p-value around **0.0086**, which would trigger an alert.

However, the public file does **not** provide the target allocation configuration.

Therefore:

> We can say that a 50/50 assumption raises an SRM concern, but we cannot prove from this file alone that the original experiment had SRM.

In production, verify the experiment configuration and assignment/exposure logs before trusting the causal result.


## 7. Statistical Analysis

After the planned horizon and outcome maturation, run the **pre-specified** test.

For Cookie Cats:

- primary metric = binary D7 retention
- one outcome per player
- sample size is large
- normal approximation is appropriate

Effect definition:

`Treatment - Control = gate_40 - gate_30`

Report:

- control rate
- treatment rate
- absolute lift
- relative lift
- 95% CI
- p-value


In [ ]:
n_c = len(control)
n_t = len(treatment)

success_c = control["retention_7"].astype(int).sum()
success_t = treatment["retention_7"].astype(int).sum()

p_c = success_c / n_c
p_t = success_t / n_t

print("--- CLT / normal approximation checks ---")
print(f"Control successes   : {success_c:,}")
print(f"Control failures    : {n_c - success_c:,}")
print(f"Treatment successes : {success_t:,}")
print(f"Treatment failures  : {n_t - success_t:,}")

# Standard two-proportion Z-test
z_stat, p_value = proportions_ztest(
    count=[success_t, success_c],
    nobs=[n_t, n_c],
    alternative="two-sided",
)

absolute_lift = p_t - p_c
relative_lift = absolute_lift / p_c

# Unpooled CI for difference in proportions
se = np.sqrt(
    p_t * (1 - p_t) / n_t
    + p_c * (1 - p_c) / n_c
)
critical = norm.ppf(0.975)

ci_low = absolute_lift - critical * se
ci_high = absolute_lift + critical * se

print("\n--- Primary result: gate_40 - gate_30 ---")
print(f"Control gate_30    : {p_c:.4%}")
print(f"Treatment gate_40  : {p_t:.4%}")
print(f"Absolute lift      : {absolute_lift*100:+.3f} pp")
print(f"Relative lift      : {relative_lift:+.2%}")
print(f"95% CI             : [{ci_low*100:+.3f}, {ci_high*100:+.3f}] pp")
print(f"Z-statistic        : {z_stat:.4f}")
print(f"p-value            : {p_value:.6f}")

--- CLT / normal approximation checks ---
Control successes   : 8,502
Control failures    : 36,198
Treatment successes : 8,279
Treatment failures  : 37,210

--- Primary result: gate_40 - gate_30 ---
Control gate_30    : 19.0201%
Treatment gate_40  : 18.2000%
Absolute lift      : -0.820 pp
Relative lift      : -4.31%
95% CI             : [-1.328, -0.312] pp
Z-statistic        : -3.1644
p-value            : 0.001554


### How to interpret the primary result

Using all observed randomized assignments:

| Output | Approximate value |
|---|---:|
| `gate_30` D7 retention | **19.02%** |
| `gate_40` D7 retention | **18.20%** |
| Absolute lift (`40 - 30`) | **-0.82pp** |
| Relative lift | **-4.3%** |
| 95% CI | **[-1.33pp, -0.31pp]** |
| p-value | **0.0016** |

### Interpretation

The confidence interval is below zero and the p-value is well below 0.05.

If experiment integrity is valid:

> Assignment to `gate_40` lowers D7 retention relative to `gate_30`.

### p-value

The p-value is:

> the probability, under H0, of observing a result at least this extreme.

It is **not** the probability that H0 is true.

### Business threshold / MDE

The illustrative MDE was **1pp**.

The point estimate is about **-0.82pp**, while the CI spans effects smaller and larger than 1pp in magnitude.

Therefore:

> Direction is statistically clear, but the exact business magnitude relative to a 1pp threshold is not fully resolved.


## 8. Business Decision & Secondary / Guardrail Outcomes

A/B testing does not end at `p < 0.05`.

### Decision guide

| Result | Interpretation |
|---|---|
| CI excludes 0 and is entirely beyond business threshold | Strong evidence of a meaningful effect |
| p < 0.05 but CI overlaps business threshold | Statistically significant; practical magnitude uncertain |
| CI includes 0 and meaningful positive/negative values | Inconclusive |
| CI is narrow inside a predefined negligible-effect region | Evidence that any effect is too small to matter |
| Primary improves but a critical guardrail breaches harm limit | Do not ship without resolving the trade-off |
| SRM / tracking issue unresolved | Do not trust causal conclusion |
| p > 0.05 with wide CI | “No evidence” is not “no effect” |

### Guardrails

For real production experiments, define guardrail metrics and acceptable harm limits **before launch**.

A non-significant p-value on a guardrail does **not** prove safety.

Instead ask:

> Does the confidence interval rule out an unacceptable deterioration?

### Cookie Cats limitation

The public dataset does not document an official guardrail.

`sum_gamerounds` is better treated here as an **exploratory secondary engagement outcome**.


In [ ]:
# Exploratory secondary outcome: sum_gamerounds
# First inspect the distribution BEFORE choosing how much weight to give a mean-based comparison.

summary = df.groupby("version")["sum_gamerounds"].agg(
    count="count",
    mean="mean",
    median="median",
    std="std",
    max="max",
)

display(summary)

print("\nSelected quantiles:")
display(
    df.groupby("version")["sum_gamerounds"]
      .quantile([0.50, 0.90, 0.99, 0.999])
      .unstack()
)

print(
    "\nImportant: extreme values can strongly influence the mean. "
    "Do not automatically delete them. First determine whether they are valid behavior "
    "or a confirmed data-quality problem using a rule that is not chosen from the treatment result."
)

,count,mean,median,std,max
version,,,,,
gate_30,44700,52.456264,17.0,256.716423,49854
gate_40,45489,51.298776,16.0,103.294416,2640



Selected quantiles:


,0.500,0.900,0.990,0.999
version,,,,
gate_30,17.0,135.0,493.00,1052.612
gate_40,16.0,134.0,492.12,1086.096



Important: extreme values can strongly influence the mean. Do not automatically delete them. First determine whether they are valid behavior or a confirmed data-quality problem using a rule that is not chosen from the treatment result.


### Why inspect `sum_gamerounds` first?

Count / engagement metrics can be extremely skewed and contain large outliers.

If the business estimand is the **mean gamerounds per player**, Welch's t-test can still target the mean difference, especially with large samples, but:

- extreme values can dominate the point estimate
- uncertainty can become large
- blindly deleting an outlier after seeing it can bias the analysis
- a confirmed instrumentation error should be handled using a defensible data-quality rule
- bootstrap sensitivity can be useful, but bootstrap does not make bad data valid

For this notebook, D7 retention remains the confirmatory primary result.


### Cookie Cats business conclusion

A careful conclusion is:

> If the actual assignment/exposure configuration validates the experiment, `gate_40` produces lower D7 retention than `gate_30`.

Before a production rollout decision, also verify:

- the real target allocation behind the SRM check
- treatment exposure / logging quality
- predefined guardrails
- whether the estimated effect is large enough to matter operationally

The public file supports the primary retention comparison, but it does not contain every piece of production experiment metadata needed for a full launch decision.


## 9. Advanced Extensions

### 9.1 CUPED

CUPED reduces variance using a **pre-treatment** covariate correlated with the outcome.

Valid CUPED variable:

- measured before treatment
- predicts the outcome
- cannot be caused by treatment

Examples:

- previous-week sessions
- previous-month spend
- historical order frequency
- historical engagement

### Cookie Cats

`sum_gamerounds` is measured during the experiment and can itself be affected by gate position.

Therefore it is **post-treatment** and should **not** be used for CUPED.

Because this dataset does not contain a valid pre-treatment covariate, a valid CUPED estimate cannot be computed from the public file.

### Reusable CUPED template


In [ ]:
def cuped_adjust(y, x_pre):
    """
    CUPED adjustment for a VALID pre-treatment covariate x_pre.

    y     : outcome
    x_pre : covariate measured before treatment assignment
    """
    y = pd.Series(y, dtype=float)
    x_pre = pd.Series(x_pre, dtype=float)

    theta = np.cov(y, x_pre, ddof=1)[0, 1] / np.var(x_pre, ddof=1)
    y_cuped = y - theta * (x_pre - x_pre.mean())

    return y_cuped, theta

print("Template ready.")
print("Not applied to Cookie Cats because no valid pre-treatment covariate is available.")

Template ready.
Not applied to Cookie Cats because no valid pre-treatment covariate is available.


### 9.2 Multiple Testing

Multiple testing appears when you test many metrics, segments or hypotheses.

If 20 independent null hypotheses are each tested at alpha = 5%, the chance of seeing at least one false positive is much larger than 5%.

Practical approaches:

- define **one primary metric**
- use **Holm / Bonferroni** for a small family of confirmatory tests
- use **FDR** when controlling the expected false-discovery proportion is appropriate
- label broad post-hoc exploration as exploratory

### 9.3 Heterogeneous Treatment Effects (HTE)

ATE asks:

> What is the average treatment effect?

HTE asks:

> Does the treatment effect vary across user types?

Examples:

- new vs veteran users
- platform
- country
- historical engagement

Pre-specify important segments when possible. Searching many subgroups and reporting only the strongest result creates a multiple-testing problem.


## 10. Common A/B Testing Pitfalls

| Term | Meaning | Better practice |
|---|---|---|
| **Peeking / Optional Stopping** | Repeatedly checking results and stopping when `p < 0.05` | Fixed horizon or valid sequential testing |
| **P-hacking** | Trying different tests, filters or metrics until something becomes significant | Pre-specify the analysis |
| **HARKing** | Hypothesizing After the Results are Known | Define hypotheses before seeing results |
| **Multiple Testing** | Testing many metrics or segments increases false positives | Predefine primary metric; adjust when needed |
| **Sample Ratio Mismatch (SRM)** | Observed split differs unexpectedly from planned allocation | Investigate assignment, eligibility and logging |
| **Post-treatment Bias** | Adjusting for variables affected by treatment | Use only pre-treatment covariates |
| **Novelty Effect** | Users react differently simply because the experience is new | Run long enough to see whether the effect stabilizes |
| **Attrition Bias** | Missing users differ across treatment groups | Investigate differential missingness |
| **Interference / SUTVA Violation** | One user's treatment affects another user's outcome | Use an appropriate randomization unit / cluster design |
| **Statistical vs Practical Significance** | `p < 0.05` but the effect is too small to matter | Compare effect size and CI with the MDE/business threshold |

---

## Senior A/B Testing Interview Checklist

1. **Hypothesis**
2. **Primary + guardrail metrics**
3. **Randomization unit**
4. **Sample size + duration**
   - baseline
   - MDE
   - alpha
   - power
   - seasonality
   - maturation
5. **Pre-analysis plan**
   - test / estimand
   - CLT assumptions
   - fixed horizon vs sequential
   - CUPED if valid
   - multiple testing / HTE
6. **Launch + collect data**
7. **Validate experiment integrity**
   - SRM
   - exposure
   - tracking
   - missingness
   - contamination
8. **Run final analysis**
   - control / treatment
   - absolute lift
   - relative lift
   - 95% CI
   - p-value
9. **Make business decision**
   - practical significance
   - guardrails
   - uncertainty
   - experiment validity

### Mental model

**Design → Launch → Validate → Estimate → Decide**
